# GenAI Model In A Graph

`pyneat.genai.graphs.vision_language(...)` and `pyneat.genai.graphs.speech_transcriber(...)`
wrap a deployed LLiMa GenAI model as a **Neat `Graph` fragment** — a stage with named inputs and
named outputs that you compose into a larger graph, exactly like `rtsp_decoded_input` or
`video_sender`.

This notebook answers four practical questions:

- When should GenAI be a **graph fragment** and when should it stay a **direct model call**?
- What are the fragment's named inputs and outputs, and what do you push and pull on each?
- What does every field in `VisionLanguageOptions` do?
- How do you stream tokens out of a graph and read the completion metadata?

The GenAI model handle (`VisionLanguageModel`, `ASRModel`, `GenAIModel`) is covered by the core
GenAI tutorials. This notebook is only about **putting one inside a `Graph`**. It mirrors the core
tutorial `022_compose_genai_into_graph` and the public docs page
[*Compose GenAI into a Graph*](https://developer.sima.ai/software/tutorials/compose-genai-into-graph).

## Mental Model

A GenAI graph fragment is the mirror image of the model handle you already know. The model handle
is **request in, result out** on one thread. The fragment is **push on a named input, pull on a
named output**, and it lives inside a `Graph` alongside every other Neat stage.

```text
direct model :  request  ->  VisionLanguageModel.run()  ->  result   (blocking, one call)
fragment     :  push("image") + push("prompt")  ->  [ genai stage ]  ->  pull("tokens") ... pull("done")
```

The vision-language fragment exposes these endpoints (verified in
`core/src/genai/GraphFragments.cpp`):

```text
                     +-------------------------------+
   push "prompt"  -> |                               | -> pull "tokens"   (one text sample per chunk)
   push "image"   -> |    vision_language stage      | -> pull "done"     (bundle: text + metrics)
   push "use_cached_image" -> |                      | -> pull "encoded"  (bundle: image cache state)
                     |                               | -> pull "error"    (text sample on failure)
                     +-------------------------------+
```

The speech transcriber fragment exposes `audio` / `audio_path` inputs and `tokens` / `done` /
`error` outputs. Same shape, different modality.

## When To Use A Fragment Instead Of The Model Handle

Most GenAI application code should start with the **direct model API** — `VisionLanguageModel.run()`
or `.stream()`. It is simpler, it is synchronous, and there is nothing to wire.

Reach for the **graph fragment** only when GenAI has to sit *beside* other Neat work:

| Use the direct model handle when… | Use the graph fragment when… |
| --- | --- |
| You want a request → response answer. | GenAI is one stage among several in a graph. |
| The caller is ordinary Python control flow. | You want to push/pull GenAI by **named endpoint**. |
| There is no other Neat stage in the loop. | You route the same frames to a detector *and* a VLM. |
| You are prototyping. | You want application-level orchestration and back-pressure. |

The demo app [`apps/detection-vlm-assistant`](../../apps/detection-vlm-assistant/README.md) is a
worked example of the *first* column: it keeps a plain `VisionLanguageModel.run()` on a background
worker beside a detection graph. This notebook teaches the *second* column.

## Imports

Run this notebook from the DevKit `pyneat` environment. It needs a deployed VLM model directory on
the board (see **Notebook Settings**), so it will only fully execute on Modalix.

In [1]:
import numpy as np
import cv2
import pyneat as neat

print("pyneat version:", getattr(neat, "__version__", "unknown"))
print("genai submodule present:", hasattr(neat, "genai"))

pyneat version: 0.2.2
genai submodule present: True


## Notebook Settings

| Setting | Meaning |
| --- | --- |
| `MODEL_DIR` | DevKit path to a deployed LLiMa **VLM** model directory. |
| `IMAGE_PATH` | An image the VLM will describe. Any JPEG/PNG the board can read. |
| `SYSTEM_PROMPT` | Steers the assistant. Baked into the fragment at build time. |
| `USER_PROMPT` | The question pushed on the `prompt` input. |
| `MAX_NEW_TOKENS` | Upper bound on generated tokens. `0` uses the model default. |

Download a VLM on the DevKit with the LLiMa CLI before running, e.g.:

```bash
llima pull Qwen3-VL-4B-Instruct-GPTQ-a16w4
```

…then point `MODEL_DIR` at the pulled directory (commonly under `/media/nvme/llima/models/`).

In [2]:
MODEL_DIR = "/media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4"  # deployed VLM directory on the DevKit
IMAGE_PATH = "../assets/images/image1.png"  # replace with an image on the board

SYSTEM_PROMPT = "You are concise."
USER_PROMPT = "Describe this image in one sentence."
MAX_NEW_TOKENS = 96

print("model:", MODEL_DIR)
print("image:", IMAGE_PATH)

model: /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4
image: ../assets/images/image1.png


## Public Fields

As in the II-medium notebooks, start from discovery rather than a hand-written list. The GenAI
surface lives under the `neat.genai` submodule; the graph fragment factories live under
`neat.genai.graphs`.

In [3]:
def public_names(obj):
    return [name for name in dir(obj) if not name.startswith("_")]


print("neat.genai:")
print(public_names(neat.genai))
print()
print("neat.genai.graphs:")
print(public_names(neat.genai.graphs))
print()
print("VisionLanguageOptions fields:")
print(public_names(neat.genai.VisionLanguageOptions()))

neat.genai:
['ASRModel', 'ChatMessage', 'GenAIModel', 'GenAIServer', 'GenAIServerOptions', 'GenAITask', 'GenerationMetrics', 'GenerationRequest', 'GenerationResult', 'GenerationStream', 'ImageList', 'SpeechTranscriberOptions', 'TokenSample', 'VisionLanguageModel', 'VisionLanguageOptions', 'graphs']

neat.genai.graphs:
['speech_transcriber', 'vision_language']

VisionLanguageOptions fields:
['encode_images_on_input', 'max_new_tokens', 'streaming', 'system_prompt']


## `VisionLanguageOptions` Field Map

These four fields are baked into the fragment when you build it. Source of truth:
`core/include/genai/GenAIOptions.h`.

| Field | Type / default | What it does |
| --- | --- | --- |
| `system_prompt` | `str`, `""` | System steer applied to every generation from this stage. |
| `max_new_tokens` | `int`, `0` | Upper bound on generated tokens. `0` = model default. |
| `streaming` | `bool`, `True` | Emit `tokens` chunk-by-chunk. `False` emits only once, at `done`. |
| `encode_images_on_input` | `bool`, `True` | Cache image embeddings as images arrive, so a later prompt can reuse them without re-encoding. Set `False` to attach the most recent image to the next prompt directly (what this notebook does). |

`streaming = True` is what makes the `tokens` output produce partial text as the model generates,
instead of one block at the end. It is the reason a graph fragment is worth the wiring for chat-like
UX.

## The Fragment's Named Endpoints

A fragment keeps its public endpoint names when you `add()` it to a bigger graph, so application
code pushes and pulls **by name**. Source of truth: `core/src/genai/GraphFragments.cpp`.

### Vision-Language inputs

| Input | Push a… | Meaning |
| --- | --- | --- |
| `prompt` | text sample | The user turn. Pushing it triggers a generation. |
| `image` | RGB image sample | uint8 HWC RGB image for the VLM to look at. |
| `use_cached_image` | text sample | Ask the stage to reuse previously encoded image embeddings. |

### Vision-Language outputs

| Output | Pull a… | Meaning |
| --- | --- | --- |
| `tokens` | text sample | One chunk of generated text (streaming) or the whole answer (non-streaming). |
| `done` | **bundle** sample | End-of-generation marker with `text`, `finish_reason`, and metrics. |
| `encoded` | bundle sample | Image-cache state: `mode`, `image_count`, `cached_image_count`. |
| `error` | text sample | Emitted instead of `done` when generation fails. |

The pull order that matters: **drain `tokens` first, and treat a `done` (or `error`) as the signal
to stop.**

## Step 1 — Create The Fragment

Load a task-specific model handle, configure `VisionLanguageOptions`, then build a public `Graph`
fragment from it. `encode_images_on_input = False` keeps this example simple: the image we push is
attached to the next prompt directly.

In [4]:
model = neat.genai.VisionLanguageModel(MODEL_DIR)

options = neat.genai.VisionLanguageOptions()
options.system_prompt = SYSTEM_PROMPT
options.max_new_tokens = MAX_NEW_TOKENS
options.streaming = True
options.encode_images_on_input = False

genai_fragment = neat.genai.graphs.vision_language(model, options, "genai_stage")
print("built fragment:", type(genai_fragment).__name__)

Loading model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_vision_stage1_mla.elf
Done loading /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_vision_stage1_mla.elf
built fragment: Graph


[mlatiming] mlashm_load_models bulk took 4011 ms ok=1
[mlatiming] mlashm_load_model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_vision_stage1_mla.elf took 2905 ms ok=1


## Step 2 — Compose It Into An App Graph

`add()` drops the fragment into a larger `Graph`. In a real app this is where you would also add a
detector, a `video_sender`, or a `metadata_sender` and wire them together. Here the fragment is the
whole graph. `describe()` prints the composed graph's structure — the graph-level view, not the
GStreamer backend string.

In [5]:
app = neat.Graph("genai_app")
app.add(genai_fragment)

print(app.describe())

Graph "genai_app" {
  nodes:
    n0: Input label="prompt" input="prompt"
    n1: Input label="image" input="image"
    n2: Input label="use_cached_image" input="use_cached_image"
    n3: GenAIVisionLanguage label="vision_language"
    n4: Output label="tokens" output="tokens"
    n5: Output label="done" output="done"
    n6: Output label="encoded" output="encoded"
    n7: Output label="error" output="error"
  edges:
    n0 -> n3 (runtime-port out -> prompt)
    n1 -> n3 (runtime-port out -> image)
    n2 -> n3 (runtime-port out -> use_cached_image)
    n3 -> n4 (runtime-port tokens -> in)
    n3 -> n5 (runtime-port done -> in)
    n3 -> n6 (runtime-port encoded -> in)
    n3 -> n7 (runtime-port error -> in)
}



## Step 3 — Build The Run

`build()` compiles the graph into a `Run` — the push/pull handle. From here on you interact only
through named endpoints.

In [6]:
run = app.build()
print("run built; last_error:", run.last_error() or "(none)")

[1/4] Initializing runtime...


run built; last_error: (none)


[2/4] Building graph...
[3/4] Starting pipeline...
[4/4] Graph ready (70 ms)


## Build The Image Sample

The VLM wants a **uint8 HWC RGB** image tensor. OpenCV loads BGR, so convert. `Tensor.from_numpy`
with `image_format=neat.PixelFormat.RGB` tags the tensor correctly; `make_tensor_sample` names it
for the `image` endpoint.

In [7]:
def load_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"failed to read image: {path}")
    return np.asarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))


rgb = load_rgb(IMAGE_PATH)
image_tensor = neat.Tensor.from_numpy(rgb, copy=True, image_format=neat.PixelFormat.RGB)
image_sample = neat.make_tensor_sample("image", image_tensor)
print("image sample ready:", rgb.shape, rgb.dtype)

image sample ready: (400, 600, 3) uint8


## Step 4 — Push The Image, Then The Prompt

Push the image first so it is available when the prompt arrives, then push the prompt. Pushing the
**prompt** is what starts a generation. `push()` returns `False` on a contract mismatch — always
check it and read `run.last_error()`.

In [8]:
if not run.push("image", [image_sample]):
    raise RuntimeError(f"push(image) failed: {run.last_error()}")

prompt_sample = neat.make_text_sample("prompt", USER_PROMPT)
if not run.push("prompt", [prompt_sample]):
    raise RuntimeError(f"push(prompt) failed: {run.last_error()}")

print("pushed image + prompt")

pushed image + prompt


## Step 5 — Stream Tokens Until `done`

The loop is the same shape as the core tutorial: pull `tokens` and print each chunk; if there is no
token, check `done` (stop) and `error` (raise). A single text sample carries its text via
`sample.to_text()`.

In [9]:
def sample_text(sample):
    """Best-effort text of a single text sample; empty string if it is not one."""
    try:
        return sample.to_text()
    except Exception:
        return ""


print("assistant: ", end="", flush=True)
done_sample = None
for _ in range(512):
    token = run.pull("tokens", 250)
    if token is not None:
        print(sample_text(token), end="", flush=True)
        continue
    done_sample = run.pull("done", 10)
    if done_sample is not None:
        break
    error = run.pull("error", 10)
    if error is not None:
        raise RuntimeError(sample_text(error))
print("\n")

assistant: People navigate a rainy city street, crossing a wet intersection while pedestrians and vehicles move through the downpour.



## Reading The `done` Bundle

`done` is a **bundle** sample, not a plain text sample. Each field is itself a named text sample.
The fields (from `core/src/genai/nodes/VisionLanguageNode.cpp`):

| Field | Meaning |
| --- | --- |
| `text` | The full generated answer. |
| `finish_reason` | `stop`, `interrupted`, `cache_full`, … |
| `generated_tokens` | Token count for this generation. |
| `time_to_first_token_s` | Latency to the first token. |
| `tokens_per_second` | Throughput. |
| `cached_image_count` | Images currently held in the stage's embedding cache. |

Walk `sample.fields` and key by each field's `stream_label` to read them individually.

In [10]:
def bundle_fields(sample):
    """Map a bundle sample to {field_name: text}."""
    out = {}
    for field in getattr(sample, "fields", []):
        label = getattr(field, "stream_label", "") or getattr(field, "port_name", "")
        out[label] = sample_text(field)
    return out


if done_sample is not None:
    fields = bundle_fields(done_sample)
    print("finish_reason      :", fields.get("finish_reason"))
    print("generated_tokens   :", fields.get("generated_tokens"))
    print("time_to_first_token:", fields.get("time_to_first_token_s"))
    print("tokens_per_second  :", fields.get("tokens_per_second"))
    print("cached_image_count :", fields.get("cached_image_count"))
else:
    print("no done sample captured")

finish_reason      : stop
generated_tokens   : 22
time_to_first_token: 0.66951632999999999
tokens_per_second  : 14.16050728261437
cached_image_count : 0


## Ask A Second Question On The Same Run

The `Run` is reusable. Push a new `prompt` to start another generation. Whether the previous image
still applies depends on the stage's image cache and `encode_images_on_input`; push a fresh `image`
when you want the model to look at something new.

In [11]:
def ask(run, question, timeout_ms=250):
    if not run.push("prompt", [neat.make_text_sample("prompt", question)]):
        raise RuntimeError(f"push(prompt) failed: {run.last_error()}")
    chunks = []
    for _ in range(512):
        token = run.pull("tokens", timeout_ms)
        if token is not None:
            chunks.append(sample_text(token))
            continue
        if run.pull("done", 10) is not None:
            break
        err = run.pull("error", 10)
        if err is not None:
            raise RuntimeError(sample_text(err))
    return "".join(chunks)


# answer = ask(run, "What is the dominant color?")
# print(answer)

## Stop The Run

Close the run when you are done so the stage and its model are released. Never leave a GenAI run
open across notebook cells you are not actively using — the model holds MLA memory.

In [12]:
run.close()
print("run closed")

run closed


## Pattern — A Text-Only LLM In A Graph

The same fragment factory serves a text-only LLM: a `VisionLanguageModel` whose
`accepts_image()` is `False`. Skip the `image` push and use only `prompt` / `tokens` / `done`.
The endpoints are identical; you simply never push `image`.

In [13]:
# llm = neat.genai.VisionLanguageModel(TEXT_LLM_DIR)
# print("accepts_image:", llm.accepts_image())  # False for a text-only LLM
#
# opt = neat.genai.VisionLanguageOptions()
# opt.system_prompt = "You are a terse assistant."
# opt.max_new_tokens = 128
# frag = neat.genai.graphs.vision_language(llm, opt, "llm_stage")
#
# g = neat.Graph("llm_app"); g.add(frag); r = g.build()
# r.push("prompt", [neat.make_text_sample("prompt", "List three primary colors.")])
# ... pull "tokens" until "done" ... then r.close()

## Pattern — Cached Image Embeddings

Encoding an image is the expensive part of a VLM turn. If you will ask **several questions about
the same image**, encode it once and reuse the embeddings.

- In a **fragment**, set `encode_images_on_input = True`, push the `image` once, then push each
  `prompt`; the stage reuses the cached embedding. Push the `use_cached_image` input to control
  reuse explicitly.
- With the **direct model handle**, the same idea is `model.encode(images)` followed by
  `request.use_cached_images = True` on each `GenerationRequest`.

The direct-handle form is shown below because its API is fully synchronous and easy to reason about.

In [14]:
# vlm = neat.genai.VisionLanguageModel(MODEL_DIR)
# vlm.encode([image_tensor])            # encode once; embeddings are cached in the model
# print("cached images:", vlm.cached_image_count())
#
# for question in ["What objects are visible?", "Is it day or night?"]:
#     req = neat.genai.GenerationRequest()
#     req.prompt = question
#     req.use_cached_images = True       # reuse the embedding instead of re-encoding
#     req.max_new_tokens = 64
#     print(question, "->", vlm.run(req).text)

## Pattern — A Speech Transcriber Fragment

Audio has its own fragment with the same push/pull shape. `speech_transcriber` exposes `audio` /
`audio_path` inputs and `tokens` / `done` / `error` outputs, configured with
`SpeechTranscriberOptions` (`language`, `streaming`).

In [15]:
# asr = neat.genai.ASRModel(ASR_MODEL_DIR)
# asr_opt = neat.genai.SpeechTranscriberOptions()
# asr_opt.language = "en"
# asr_opt.streaming = True
#
# asr_frag = neat.genai.graphs.speech_transcriber(asr, asr_opt, "asr_stage")
# g = neat.Graph("asr_app"); g.add(asr_frag); r = g.build()
# r.push("audio_path", [neat.make_text_sample("audio_path", "/path/to/clip.wav")])
# ... pull "tokens" until "done" ... then r.close()

## The Direct Model API, For Contrast

So the trade-off is concrete: here is the **same answer** with no graph at all. If your code looks
like this and there is no other Neat stage in the loop, you do **not** need a fragment.

In [16]:
# vlm = neat.genai.VisionLanguageModel(MODEL_DIR)
#
# req = neat.genai.GenerationRequest()
# req.system_prompt = SYSTEM_PROMPT
# req.prompt = USER_PROMPT
# req.images = [image_tensor]
# req.max_new_tokens = MAX_NEW_TOKENS
#
# result = vlm.run(req)                 # blocking; one call
# print(result.text)
# print(result.finish_reason, result.metrics.tokens_per_second)
#
# # Streaming variant:
# for token in vlm.stream(req):
#     print(token.text, end="", flush=True)
#     if token.is_final:
#         break

## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `VisionLanguageModel(...)` raises | `MODEL_DIR` is not a deployed LLiMa model directory. | `llima pull <model>`; point `MODEL_DIR` at the pulled folder. |
| `push("image", ...)` returns `False` | Image is not uint8 HWC RGB. | Convert BGR→RGB and build with `image_format=neat.PixelFormat.RGB`. |
| `push("prompt", ...)` returns `False` | Endpoint name typo, or the run was closed. | Use the exact names `prompt` / `image`; rebuild the run. |
| `pull("tokens")` always `None`, no `done` | Prompt was never pushed, so nothing is generating. | Pushing `prompt` is what starts a generation. |
| Generation never streams | `options.streaming = False`. | Set `streaming = True` to get chunked `tokens`. |
| `to_text()` raises on `done` | `done` is a **bundle**, not a text sample. | Read fields with `bundle_fields(done)`, not `done.to_text()`. |
| An `error` sample arrives | The model rejected the request. | `sample_text(error)` carries the message; check prompt/image validity. |
| Board runs out of MLA memory | A previous run was never closed. | `run.close()` when done; do not hold runs open across idle cells. |

## What To Remember

- A GenAI fragment is a `Graph` stage with **named endpoints**, built by
  `neat.genai.graphs.vision_language(model, options, name)`.
- Prefer the **direct model handle** (`VisionLanguageModel.run()` / `.stream()`) for plain
  request/response. Use the **fragment** only when GenAI sits beside other Neat stages.
- Vision-language inputs are `prompt`, `image`, `use_cached_image`; outputs are `tokens`, `done`,
  `encoded`, `error`. Speech uses `audio` / `audio_path` → `tokens` / `done` / `error`.
- Pushing the `prompt` is what triggers a generation. Push the `image` first.
- Drain `tokens`, and treat `done` (or `error`) as the stop signal.
- `done` is a **bundle**: `text`, `finish_reason`, `generated_tokens`, `time_to_first_token_s`,
  `tokens_per_second`, `cached_image_count`.
- `streaming = True` is what makes `tokens` chunked; `encode_images_on_input` controls the image
  embedding cache.
- Images are uint8 HWC **RGB** tensors — convert from OpenCV BGR first.
- `run.close()` releases the model. GenAI models are heavy; do not leak runs.

## References

- Public docs: [*Compose GenAI into a Graph*](https://developer.sima.ai/software/tutorials/compose-genai-into-graph).
- Core tutorial: `core/tutorials/022_compose_genai_into_graph/` (`.py` and `.cpp`).
- Core headers: `include/genai/GraphFragments.h`, `include/genai/GenAIOptions.h`,
  `include/genai/VisionLanguageModel.h`, `include/genai/GenAITypes.h`.
- Fragment wiring source of truth: `core/src/genai/GraphFragments.cpp`,
  `core/src/genai/nodes/VisionLanguageNode.cpp`.
- `pyneat` bindings: `core/python/src/module.cpp` (the `neat.genai` submodule).
- Demo app (direct model handle beside a graph):
  [`apps/detection-vlm-assistant`](../../apps/detection-vlm-assistant/README.md).